# Maximum Margin Classification

**Companion lesson:** https://ml-viz.vercel.app/courses/svm/01-maximum-margin

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Maximum Margin Concept

The SVM finds the hyperplane $w^T x + b = 0$ with the widest margin: $\text{margin} = 2 / \|w\|$

In [ ]:
np.random.seed(42)
X_pos = np.array([[2, 3], [3, 3], [3, 2], [4, 3], [3, 4]])
X_neg = np.array([[0, 0], [1, 0], [0, 1], [1, 1], [0, 2]])
X = np.vstack([X_pos, X_neg])
y = np.array([1]*5 + [-1]*5)

# SVM boundary: w = [1, 1], b = -2.5
w = np.array([1, 1])
b = -2.5

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_pos[:, 0], X_pos[:, 1], c='#818cf8', s=60, zorder=5, label='Class +1')
ax.scatter(X_neg[:, 0], X_neg[:, 1], c='#f43f5e', s=60, zorder=5, label='Class -1')

x_line = np.linspace(-1, 5, 100)
ax.plot(x_line, (-b - w[0]*x_line) / w[1], color='#14b8a6', linewidth=2, label='Decision boundary')
ax.plot(x_line, (-b - 1 - w[0]*x_line) / w[1], '--', color='#14b8a6', alpha=0.5, label='Margin')
ax.plot(x_line, (-b + 1 - w[0]*x_line) / w[1], '--', color='#14b8a6', alpha=0.5)
ax.legend()
ax.set_title(f'Maximum Margin Classifier (margin = {2/np.linalg.norm(w):.2f})', color='white')
plt.tight_layout()
plt.show()

## Solving for the max-margin hyperplane by hand

The margin is $2/\lVert w\rVert$ because the signed distance from a point to $w^\top x+b=0$ is $(w^\top x+b)/\lVert w\rVert$, and we rescale so the support vectors sit at $w^\top x+b=\pm1$ — giving a street of half-width $1/\lVert w\rVert$ on each side. Below we solve the lesson's 6-point example: the support vectors $(2,2)$ and $(0,0)$ fix two equations, yielding $w=(0.5,0.5)$, $b=-1$, margin $=2\sqrt2$ — then `sklearn` confirms it.

In [ ]:
import numpy as np
from sklearn.svm import SVC

Xs = np.array([[2, 2], [3, 3], [4, 4], [0, 0], [-1, -1], [-2, -2]], float)
ys = np.array([1, 1, 1, -1, -1, -1])

# By hand: support vectors are the closest opposing pair (2,2) and (0,0).
# Boundary normal w ∝ (2,2)-(0,0) = (1,1); write w = a*(1,1).
# Margin equations:  w·(2,2)+b = +1  ->  4a + b = 1
#                    w·(0,0)+b = -1  ->        b = -1
b = -1.0
a = (1 - b) / 4          # = 0.5
w = a * np.array([1.0, 1.0])
margin = 2 / np.linalg.norm(w)
print(f'by hand : w={w}, b={b}, margin={margin:.3f}  (2*sqrt(2)={2*np.sqrt(2):.3f})')

# Functional margins y_i (w·x_i + b): SVs hit exactly 1, others strictly >1
print('y*(w·x+b):', np.round(ys * (Xs @ w + b), 3))

# Distance between the two support vectors == full margin width
print('dist((2,2),(0,0)) =', round(np.linalg.norm(np.array([2, 2.0])), 3))

# sklearn cross-check (hard margin via large C)
svm = SVC(kernel='linear', C=1000).fit(Xs, ys)
print(f'\nsklearn : w={np.round(svm.coef_[0], 3)}, b={svm.intercept_[0]:.3f}, '
      f'margin={2/np.linalg.norm(svm.coef_[0]):.3f}')
print('support vectors:\n', svm.support_vectors_)


## Hinge Loss

In [ ]:
z = np.linspace(-3, 3, 200)
hinge = np.maximum(0, 1 - z)
logistic = np.log(1 + np.exp(-z))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z, hinge, color='#818cf8', linewidth=2, label='Hinge loss')
ax.plot(z, logistic, color='#14b8a6', linewidth=2, label='Logistic loss')
ax.axvline(0, color='#2e3347', linestyle='--')
ax.set_title('SVM Hinge Loss vs Logistic Loss', color='white')
ax.set_xlabel('y · f(x)')
ax.legend()
plt.tight_layout()
plt.show()

## Soft margin: the C parameter

$C$ controls the trade-off between a **wide margin** and **few violations**. Small $C$ = wider margin, more slack; large $C$ = fewer errors, narrower margin.

In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=80, centers=2, cluster_std=1.8, random_state=6)
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-1, X[:,0].max()+1, 200),
                     np.linspace(X[:,1].min()-1, X[:,1].max()+1, 200))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, C in zip(axes, [0.05, 1, 100]):
    svm = SVC(kernel='linear', C=C).fit(X, y)
    Z = svm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z > 0, alpha=0.2, cmap='RdBu')
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors='k', linestyles=['--','-','--'])
    ax.scatter(*X.T, c=y, cmap='RdBu', edgecolor='k', s=25)
    ax.set_title(f'C = {C}  ({svm.support_.size} SVs)')
plt.tight_layout(); plt.show()

## Key takeaways

- SVM finds the separating hyperplane with the **maximum margin** to the nearest points.
- Only the **support vectors** (points on the margin) define the boundary.
- **Soft margin** ($C$) allows violations to handle noisy / overlapping data.
- **Hinge loss** is the soft-margin objective; always standardize features first.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Hinge loss

The loss behind SVMs charges nothing once a point clears the margin, and a linear penalty otherwise:

$$\ell(y, s) = \max(0, \, 1 - y \cdot s) \qquad y \in \{-1, +1\}$$

Implement it vectorized. The checks walk the regimes: confidently correct ($ys \geq 1$) → 0, inside the margin → partial penalty, wrong side → penalty $> 1$.

In [ ]:
def hinge(y, score):
    """Hinge loss for labels y in {-1, +1} and raw scores. Vectorized."""
    y = np.asarray(y, dtype=float)
    score = np.asarray(score, dtype=float)

    # TODO(you): max(0, 1 - y * score) elementwise (hint: np.maximum)
    return ...

In [ ]:
# Checks — run me
assert hinge(1, 2.0) == 0.0, "confidently correct (margin met) -> zero loss"
assert hinge(1, 1.0) == 0.0, "exactly on the margin -> zero loss"
assert hinge(1, 0.5) == 0.5, "inside the margin -> linear penalty"
assert hinge(-1, 1.0) == 2.0, "on the wrong side -> big penalty"
assert np.allclose(hinge([1, -1, 1], [2.0, -2.0, 0.0]), [0.0, 0.0, 1.0]), "vectorized"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def hinge(y, score):
    y = np.asarray(y, dtype=float)
    score = np.asarray(score, dtype=float)
    return np.maximum(0.0, 1.0 - y * score)
```

</details>

### Exercise 2 — Margin width

With the canonical scaling (support vectors at $|\mathbf{w} \cdot \mathbf{x} + b| = 1$), the gap between the two margin lines is

$$\text{width} = \frac{2}{\lVert \mathbf{w} \rVert}$$

The last check is the whole story of SVM training in one line: **doubling $\mathbf{w}$ halves the margin** — which is exactly why maximizing the margin means minimizing $\lVert \mathbf{w} \rVert$.

In [ ]:
def margin_width(w):
    """Width of the margin band for weight vector w (canonical scaling)."""
    # TODO(you): 2 / ||w||  (hint: np.linalg.norm)
    return ...

In [ ]:
# Checks — run me
assert abs(margin_width([2.0, 0.0]) - 1.0) < 1e-12, "||w|| = 2 -> width 1"
assert abs(margin_width([1.0, 1.0]) - np.sqrt(2)) < 1e-12, "||w|| = sqrt(2) -> width sqrt(2)"
assert abs(margin_width([2.0, 2.0]) - margin_width([1.0, 1.0]) / 2) < 1e-12, \
    "doubling w halves the margin — why SVM minimizes ||w||"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def margin_width(w):
    return 2.0 / np.linalg.norm(np.asarray(w, dtype=float))
```

</details>